In [1]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [2]:
CHECKPOINT_DIR = Path("notebooks/models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "leaderboard_week3_best.pt"
print("Checkpoint will be saved to:", CHECKPOINT_PATH)

Checkpoint will be saved to: notebooks\models\leaderboard_week3_best.pt


In [3]:
# Standard library
import copy
import inspect
import json
import random

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# Scikit-learn
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Neuroprobe
import neuroprobe
import neuroprobe.train_test_splits as neuroprobe_train_test_splits
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [4]:
# =========================
# Config
# =========================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASKS = ["delta_volume", "speech", "pitch", "gpt2_surprisal", "word_gap"]

BATCH_SIZE = 8
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

STFT_N_FFT = 64
STFT_HOP = 16
STFT_WIN_LEN = 32
LAPLACIAN_K = 4

SUBJ_EMB_DIM = 16
TASK_EMB_DIM = 8

COORD_DIM = 3
COORD_EMB_DIM = 16
ELEC_HIDDEN_DIM = 128
MODEL_DIM = 128

NUM_VIRTUAL_SENSORS = 16
NUM_SENSOR_HEADS = 4
ATTN_DROPOUT = 0.1
PROJ_DROPOUT = 0.1

USE_COORDS_IN_KEYS = True
USE_COORDS_IN_VALUES = False
USE_SENSOR_SELF_ATTN = True
NUM_SENSOR_SELF_ATTN_LAYERS = 1

TEST_SUBJECT_ID = 1
TEST_TRIAL_ID = 2
SUBJECT_IDS = list(range(1, 11))

USE_GLOBAL_TRAIN_NORM = False
USE_VAL_AS_TEST = False

MASK_VALUE = -1e9

print("DEVICE:", DEVICE)

DEVICE: cpu


In [5]:
# =========================
# Virtual sensor harmonizer
# =========================
class VirtualSensorHarmonizer(nn.Module):
    def __init__(
            self,
            elec_hidden_dim,
            coord_dim,
            coord_emb_dim,
            model_dim,
            num_virtual_sensors,
            num_heads,
            attn_dropout=0.1,
            proj_dropout=0.1,
            use_coords_in_keys=True,
            use_coords_in_values=False,
            use_sensor_self_attn=True,
    ):
        super().__init__()

        self.use_coords_in_keys = use_coords_in_keys
        self.use_coords_in_values = use_coords_in_values
        self.use_sensor_self_attn = use_sensor_self_attn

        self.coord_mlp = nn.Sequential(
            nn.Linear(coord_dim, coord_emb_dim),
            nn.LayerNorm(coord_emb_dim),
            nn.ReLU(),
            nn.Dropout(proj_dropout),
            nn.Linear(coord_emb_dim, coord_emb_dim),
        )

        key_in_dim = elec_hidden_dim + (coord_emb_dim if use_coords_in_keys else 0)
        val_in_dim = elec_hidden_dim + (coord_emb_dim if use_coords_in_values else 0)

        self.key_proj = nn.Linear(key_in_dim, model_dim)
        self.val_proj = nn.Linear(val_in_dim, model_dim)

        self.virtual_queries = nn.Parameter(
            torch.randn(num_virtual_sensors, model_dim) * 0.02
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,
        )

        if use_sensor_self_attn:
            self.sensor_self_attn = nn.MultiheadAttention(
                embed_dim=model_dim,
                num_heads=num_heads,
                dropout=attn_dropout,
                batch_first=True,
            )
            self.sensor_ff = nn.Sequential(
                nn.Linear(model_dim, model_dim * 2),
                nn.ReLU(),
                nn.Dropout(proj_dropout),
                nn.Linear(model_dim * 2, model_dim),
            )
            self.norm1 = nn.LayerNorm(model_dim)
            self.norm2 = nn.LayerNorm(model_dim)

        self.out_norm = nn.LayerNorm(model_dim)
        self.out_drop = nn.Dropout(proj_dropout)

    def forward(self, elec_feat, coords, elec_mask):
        coord_feat = self.coord_mlp(coords)

        if self.use_coords_in_keys:
            key_in = torch.cat([elec_feat, coord_feat], dim=-1)
        else:
            key_in = elec_feat

        if self.use_coords_in_values:
            val_in = torch.cat([elec_feat, coord_feat], dim=-1)
        else:
            val_in = elec_feat

        keys = self.key_proj(key_in)
        values = self.val_proj(val_in)

        B = elec_feat.size(0)
        queries = self.virtual_queries.unsqueeze(0).expand(B, -1, -1)

        key_padding_mask = ~elec_mask.bool()

        sensors, _ = self.cross_attn(
            query=queries,
            key=keys,
            value=values,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )

        if self.use_sensor_self_attn:
            attn_out, _ = self.sensor_self_attn(
                query=sensors,
                key=sensors,
                value=sensors,
                need_weights=False,
            )
            sensors = self.norm1(sensors + attn_out)
            ff_out = self.sensor_ff(sensors)
            sensors = self.norm2(sensors + ff_out)

        sensors = self.out_norm(self.out_drop(sensors))
        return sensors

In [6]:
# =========================
# Harmonizer smoke test
# =========================
harmonizer = VirtualSensorHarmonizer(
    elec_hidden_dim=ELEC_HIDDEN_DIM,
    coord_dim=COORD_DIM,
    coord_emb_dim=COORD_EMB_DIM,
    model_dim=MODEL_DIM,
    num_virtual_sensors=NUM_VIRTUAL_SENSORS,
    num_heads=NUM_SENSOR_HEADS,
    attn_dropout=ATTN_DROPOUT,
    proj_dropout=PROJ_DROPOUT,
    use_coords_in_keys=USE_COORDS_IN_KEYS,
    use_coords_in_values=USE_COORDS_IN_VALUES,
    use_sensor_self_attn=USE_SENSOR_SELF_ATTN,
).to(DEVICE)

B, E = 4, 11
elec_feat = torch.randn(B, E, ELEC_HIDDEN_DIM, device=DEVICE)
coords = torch.randn(B, E, COORD_DIM, device=DEVICE)
elec_mask = torch.ones(B, E, dtype=torch.bool, device=DEVICE)
elec_mask[1, -3:] = False
elec_mask[2, -5:] = False

with torch.no_grad():
    sensor_tokens = harmonizer(elec_feat, coords, elec_mask)

print("elec_feat:", elec_feat.shape)
print("coords:", coords.shape)
print("elec_mask:", elec_mask.shape, elec_mask.dtype)
print("sensor_tokens:", sensor_tokens.shape)
assert sensor_tokens.shape == (B, NUM_VIRTUAL_SENSORS, MODEL_DIM)
print("smoke test passed")

elec_feat: torch.Size([4, 11, 128])
coords: torch.Size([4, 11, 3])
elec_mask: torch.Size([4, 11]) torch.bool
sensor_tokens: torch.Size([4, 16, 128])
smoke test passed
